### Data Generators

In [ ]:


class DataGenerator:


    def data_gen(self):
        for patient in patients:
            # mesh is a customObject inherited from pv mesh object 
            img = load(patient)
            boundary_conditions = load(patient_bcs)
            if fileexists(tempdir/patient):
                out_dict = read(tempdir/patient)
            else:
                mesh = Mesh(file_list, feature_keys = ['pressure', ...], ids= {"volume":0, "surface":1, "inlets":2, "outlet1":3,"outlet2":4}, fixed_coords=True, fixed_connectivity=True)
                mesh.scale_features(feature_scaler)
                mesh.temporal_pca(PCA)
                mesh.filter_adjacency(filter_type, filter_degree, ["volume"], keep_original=False)
                mesh.compute_normals(["inlet", "outlet1" ,"outlet2"])
                out_dict = mesh.to_dict()
            yield {'img':img, 'boundary':boundary_conditions}, out_dict 
    
    def get_signature(self):
        # gets the output signature of the data generator



# The data generator has can be made by subclassing the DataGenerator and adding a custom data_gen or by using one of the presets
# The get_signature function returns the signature of the function you create 
# The inputs are formatted for your model structure, outputs are dict(points, features, pca_features and structure_adjacency,... )
# The contents of each sub category is user-specified
# Custom calculated values can be added to the final dict
# put Mesh functions in a utils file outside of the object so you can use it functionally also

In [ ]:

class Loss:
    def __init__(self, structures, name):

    def call(args):

    def get_keys(self):
        # return [structure_arg for stucture, arg in self.structures, self.call.args]

class CustomLoss(Loss):
    def call(adjacency, features):
        ...

# All losses will return keys needed for the loss based on the structure it was initialised with
# The losses are subclassed from Im2SimLoss and the call function keys are used for mapping
cap_loss = CapLoss(structures = ["inlet", "outlet1" ,"outlet2"])
surface_chamfer = ChamferLoss(structures=["surface"])
volume_chamfer = ChamferLoss(structures=["volume"])



In [ ]:
class Model(tf.keras.Model):
    def set_losses(losses):
        # self.losses = losses
    def train_step(self, data):
        x, y_true = data

        with tf.GradientTape() as tape:
            y_pred = self(x, training=True)
            # Use compiled loss (supports any loss passed in compile)
            for key in self.losses.keys(): # coords/features 
                for i, loss in enumerate(self.losses[key]):
                    keys = loss.get_keys()
                    self.loss_vals[key][i] = loss(y_pred[key], y_true[keys])
            for key in y_pred.keys():
                if 'residual' in key:
                    self.lossvals[key][i] = y_pred[key]

        gradients = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_variables))

        # Update compiled metrics
        self.compiled_metrics.update_state(y_true, y_pred)
        
        # Return a dict mapping metric names to current value
        return {m.name: m.result() for m in self.metrics}
    

image_input = Input((None,None,None,1))
enc = ImageEncoder(*args)(image_input)
template_input = Input((None, 7))
boundary_condition_input = Input((None, 7))
dec = GraphDecoder(*args)([enc, template_input, boundary_condition_input])
phys_block = NavierStokes(*args)(dec)

model = Im2SimModel(image_input, phys_block)
model.set_losses(losses)

model.compile()

model.fit()